# 03 — Free Open-Source LLM-as-Judge

**Model:** `microsoft/Phi-3-mini-4k-instruct`

This notebook evaluates hallucination detection using a zero-shot, open-source LLM-as-a-Judge.

The model receives the source context and generated response and estimates the probability of answering **No** to the question: “Is every claim in the response fully supported by the source?”

This probability, `llm_p_no`, serves as a continuous hallucination score. A higher value indicates that the model considers the response more likely to contain unsupported claims.

We will reuse the existing canonical train-validation-test split, select the classification threshold on the validation set, evaluate on the held-out test set, and calculate bootstrap confidence intervals.

The full evaluation will save checkpoints to Google Drive so that it can resume after a Colab disconnection.

In [4]:
!pip install -q transformers accelerate

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM (GB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    )
else:
    print(
        "WARNING: no GPU — switch Runtime > Change runtime type > T4 GPU."
    )

CUDA available: True
GPU: Tesla T4
VRAM (GB): 15.6


### Load the canonical evaluation dataframe and split

Purpose: This section loads our prepared dataset and the existing train-validation-test assignment. We must reuse the original split for a fair comparison with ROUGE, BERTScore, and NLI.

In [2]:
#Mount Drive and load the data
import os, ast, time
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/rag-eval-metrics'

eval_df = pd.read_csv(f'{DRIVE_DIR}/ragtruth_eval_dataframe.csv')
eval_df['labels'] = eval_df['labels'].apply(
    lambda x: [] if pd.isna(x) or x == '' else ast.literal_eval(x)
)
split_df = pd.read_csv(f'{DRIVE_DIR}/ragtruth_split_assignment.csv')

assert eval_df.shape[0] == 1781, f'Unexpected row count: {eval_df.shape[0]}'
assert split_df.shape[0] == 1781 and set(split_df['id']) == set(eval_df['id']), \
    'Split file does not match eval dataframe ids'

print('eval_df:', eval_df.shape)
print(split_df['split'].value_counts())


Mounted at /content/drive
eval_df: (1781, 11)
split
train    1068
test      357
val       356
Name: count, dtype: int64


In [5]:
#Load the Model

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='cuda')
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='cuda')
model.eval()

print('Model dtype:', model.dtype)
print('GPU memory used (GB):', round(torch.cuda.memory_allocated() / 1e9, 1))

def _token_ids(candidates):
    ids = []
    for t in candidates:
        i = tokenizer.convert_tokens_to_ids(t)
        if i is not None and i != tokenizer.unk_token_id:
            ids.append(i)
    return ids

YES_IDS = _token_ids(['▁Yes', 'Yes'])
NO_IDS = _token_ids(['▁No', 'No'])
print('Yes token ids:', YES_IDS, '| No token ids:', NO_IDS)
assert YES_IDS and NO_IDS, 'Could not find Yes/No token ids in the tokenizer vocabulary'


model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model dtype: torch.float16
GPU memory used (GB): 7.6
Yes token ids: [3869, 8241] | No token ids: [1939, 3782]


## Prompt + scoring function, with a 4-example smoke test

We ask the LLM judge whether every claim in a response is supported by its source.

Instead of relying only on the generated word, we compare the model's probability mass for **No** versus **Yes** at the first answer position. This produces a continuous hallucination score, `llm_p_no`, where a higher value indicates a greater likelihood of hallucination.

Source contexts are truncated to a maximum of 2,800 tokens when needed to fit the model's context window.

In [6]:

MAX_CONTEXT_TOKENS = 2800

PROMPT_TEMPLATE = '''You are a careful fact-checking assistant. Below is a SOURCE document and a RESPONSE that was supposed to be written using only information from that SOURCE.

SOURCE:
{context}

RESPONSE:
{response}

Question: Is every claim in the RESPONSE fully supported by the SOURCE, with nothing contradicted or made up? Answer with exactly one word: Yes or No.'''

def build_prompt(context, response):
    ctx_ids = tokenizer(context, add_special_tokens=False).input_ids
    truncated = len(ctx_ids) > MAX_CONTEXT_TOKENS
    if truncated:
        context = tokenizer.decode(ctx_ids[:MAX_CONTEXT_TOKENS])
    user_msg = PROMPT_TEMPLATE.format(context=context, response=response)
    text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_msg}], tokenize=False, add_generation_prompt=True
    )
    return text, truncated

def score_one(context, response):
    t0 = time.time()
    text, truncated = build_prompt(context, response)
    inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False).to('cuda')
    with torch.inference_mode():
        logits = model(**inputs, use_cache=False).logits[0, -1].float()
    logp = torch.log_softmax(logits, dim=-1)
    yes_lp = torch.logsumexp(logp[YES_IDS], dim=0)
    no_lp = torch.logsumexp(logp[NO_IDS], dim=0)
    return {
        'llm_p_no': torch.sigmoid(no_lp - yes_lp).item(),
        'llm_logodds_no': (no_lp - yes_lp).item(),
        'yes_no_mass': torch.exp(torch.logsumexp(torch.stack([yes_lp, no_lp]), dim=0)).item(),
        'top_token': tokenizer.convert_ids_to_tokens(int(torch.argmax(logp))),
        'context_truncated': bool(truncated),
        'prompt_tokens': int(inputs['input_ids'].shape[1]),
        'seconds': time.time() - t0,
    }

# Smoke test: 2 hallucinated + 2 faithful responses
smoke = pd.concat([
    eval_df[eval_df['has_hallucination']].head(2),
    eval_df[~eval_df['has_hallucination']].head(2),
])
for _, row in smoke.iterrows():
    out = score_one(row['source_info'], row['response'])
    print(f"task={row['task_type']:9s} true_hallucination={row['has_hallucination']!s:5s} "
          f"p_no={out['llm_p_no']:.3f} yes/no_mass={out['yes_no_mass']:.3f} "
          f"top_token={out['top_token']!r} prompt_tokens={out['prompt_tokens']} sec={out['seconds']:.2f}")


task=Data2txt  true_hallucination=True  p_no=0.706 yes/no_mass=0.991 top_token='▁No' prompt_tokens=934 sec=2.19
task=Data2txt  true_hallucination=True  p_no=0.672 yes/no_mass=0.989 top_token='▁No' prompt_tokens=1046 sec=0.45
task=Data2txt  true_hallucination=False p_no=0.378 yes/no_mass=0.994 top_token='▁Yes' prompt_tokens=860 sec=0.39
task=Data2txt  true_hallucination=False p_no=0.860 yes/no_mass=0.991 top_token='▁No' prompt_tokens=1092 sec=0.48


## Smoke-test interpretation

Check that `yes_no_mass` is close to 1 and that `top_token` is a Yes/No-type token. If the Yes/No probability mass is tiny for every example, stop and investigate before running the full evaluation.

The `sec` value is the time taken for one response. Multiply a representative time by 1,781 to estimate the full-run duration, allowing additional time for checkpointing and overhead.

## Full run (checkpointed and resumable)

This cell will score all 1,781 responses using the LLM judge.

Progress is saved to Google Drive every 100 responses in `ragtruth_llmjudge_checkpoint.csv`. If the runtime disconnects, rerunning the scoring cell will resume from the saved checkpoint and skip responses that have already been scored.

Keep the Colab runtime connected and avoid interrupting execution unless necessary.

In [7]:
#Full LLM scoring run

from tqdm.auto import tqdm

CKPT_PATH = f'{DRIVE_DIR}/ragtruth_llmjudge_checkpoint.csv'

if os.path.exists(CKPT_PATH):
    ckpt = pd.read_csv(CKPT_PATH)
    results = ckpt.to_dict('records')
    done_ids = set(ckpt['id'])
    print(f'Resuming: {len(done_ids)} responses already scored')
else:
    results, done_ids = [], set()

todo = eval_df[~eval_df['id'].isin(done_ids)]
print(f'{len(todo)} responses left to score')

for i, (_, row) in enumerate(tqdm(todo.iterrows(), total=len(todo)), start=1):
    out = score_one(row['source_info'], row['response'])
    out['id'] = row['id']
    results.append(out)
    if i % 100 == 0 or i == len(todo):
        pd.DataFrame(results).to_csv(CKPT_PATH, index=False)

scores_df = pd.DataFrame(results)
print('Scored:', scores_df.shape[0], 'responses')
print('Contexts truncated to fit the prompt window:', int(scores_df['context_truncated'].sum()))
print('Total compute time (seconds, summed per response):', round(scores_df['seconds'].sum(), 1))


Resuming: 1781 responses already scored
0 responses left to score


0it [00:00, ?it/s]

Scored: 1781 responses
Contexts truncated to fit the prompt window: 0
Total compute time (seconds, summed per response): 914.3


## Assemble and save per-response results

Merge the LLM judge scores with the canonical evaluation dataframe and the existing train/validation/test split assignment.

Save the final per-response scores to `ragtruth_llmjudge_scores.csv`. Then inspect the mean `llm_p_no` by hallucination label and task type, along with the Yes/No probability mass summary.

In [8]:

final = (
    eval_df[['id', 'source_id', 'model', 'task_type', 'has_hallucination']]
    .merge(scores_df, on='id', how='left')
    .merge(split_df, on='id', how='left')
)
assert final.shape[0] == 1781 and final['llm_p_no'].notna().all() and final['split'].notna().all()

scores_path = f'{DRIVE_DIR}/ragtruth_llmjudge_scores.csv'
final.to_csv(scores_path, index=False)
print('Saved', scores_path)

print('\nMean P(No) by has_hallucination (higher = judge thinks more hallucinated):')
print(final.groupby('has_hallucination')['llm_p_no'].mean())
print('\nMean P(No) by task_type and has_hallucination:')
print(final.groupby(['task_type', 'has_hallucination'])['llm_p_no'].mean())
print('\nYes/No probability mass — min / mean:', round(final['yes_no_mass'].min(), 3), '/', round(final['yes_no_mass'].mean(), 3))


Saved /content/drive/MyDrive/rag-eval-metrics/ragtruth_llmjudge_scores.csv

Mean P(No) by has_hallucination (higher = judge thinks more hallucinated):
has_hallucination
False    0.380423
True     0.541072
Name: llm_p_no, dtype: float64

Mean P(No) by task_type and has_hallucination:
task_type  has_hallucination
Data2txt   False                0.443627
           True                 0.546911
QA         False                0.345834
           True                 0.595430
Summary    False                0.380688
           True                 0.473739
Name: llm_p_no, dtype: float64

Yes/No probability mass — min / mean: 0.971 / 0.995


## Canonical test-set evaluation

Select the decision threshold using the validation set by maximizing F1. Freeze that threshold and apply it to the held-out test set.

Predict a response as hallucinated when `llm_p_no >= threshold`.

Report accuracy, precision, recall, F1, ROC-AUC, PR-AUC, and the confusion matrix. ROC-AUC and PR-AUC use the continuous scores rather than thresholded predictions.

In [9]:
#Validation threshold and test evaluation : It selects the threshold that maximizes F1 on the validation set, then evaluates the frozen threshold on the test set.

from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix)

val_df = final[final['split'] == 'val']
test_df = final[final['split'] == 'test']

best_thresh, best_f1 = None, -1.0
for t in np.sort(val_df['llm_p_no'].unique()):
    preds = (val_df['llm_p_no'] >= t).astype(int)
    f1 = f1_score(val_df['has_hallucination'].astype(int), preds)
    if f1 > best_f1:
        best_f1, best_thresh = f1, t
print(f'Selected threshold (P(No)): {best_thresh:.6f} | Validation F1: {best_f1:.4f}')

y_true = test_df['has_hallucination'].astype(int).values
y_score = test_df['llm_p_no'].values
y_pred = (y_score >= best_thresh).astype(int)

test_metrics = {
    'accuracy': accuracy_score(y_true, y_pred),
    'precision': precision_score(y_true, y_pred, zero_division=0),
    'recall': recall_score(y_true, y_pred),
    'f1': f1_score(y_true, y_pred),
    'roc_auc': roc_auc_score(y_true, y_score),
    'pr_auc': average_precision_score(y_true, y_score),
}
print(f'\nHeld-out test set: n={len(test_df)} | hallucinated={y_true.sum()} | faithful={(1 - y_true).sum()}')
print(f'Majority-class (always "not hallucinated") accuracy for reference: {max(y_true.mean(), 1 - y_true.mean()):.4f}')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')
print('\nConfusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_true, y_pred))


Selected threshold (P(No)): 0.314042 | Validation F1: 0.7023

Held-out test set: n=357 | hallucinated=153 | faithful=204
Majority-class (always "not hallucinated") accuracy for reference: 0.5714
  accuracy: 0.6162
  precision: 0.5323
  recall: 0.8627
  f1: 0.6584
  roc_auc: 0.6783
  pr_auc: 0.5395

Confusion matrix [[TN, FP], [FN, TP]]:
[[ 88 116]
 [ 21 132]]


## Bootstrap 95% confidence intervals + metadata row

Estimate 95% confidence intervals for the test metrics using 2,000 bootstrap resamples of the held-out test set with random seed 42.

Keep the validation-selected threshold fixed during bootstrapping. Save the confidence interval table to `llm_judge_canonical_test_bootstrap_ci.csv` and append the LLM-as-a-Judge run details to `metric_run_metadata.csv`.

In [10]:
#Bootstrap confidence intervals + save metadata: It runs 2,000 bootstrap resamples, calculates 95% confidence intervals, saves the table, and appends the LLM judge results to metadata file.


rng = np.random.default_rng(42)
n_boot, n_test = 2000, len(test_df)
boot_records = []
for _ in range(n_boot):
    idx = rng.integers(0, n_test, n_test)
    yt, ys, yp = y_true[idx], y_score[idx], y_pred[idx]
    if yt.sum() == 0 or yt.sum() == len(yt):
        continue
    boot_records.append({
        'accuracy': accuracy_score(yt, yp),
        'precision': precision_score(yt, yp, zero_division=0),
        'recall': recall_score(yt, yp, zero_division=0),
        'f1': f1_score(yt, yp, zero_division=0),
        'roc_auc': roc_auc_score(yt, ys),
        'pr_auc': average_precision_score(yt, ys),
    })
boot_df = pd.DataFrame(boot_records)
ci_table = pd.DataFrame({
    'estimate': pd.Series(test_metrics),
    'ci_low': boot_df.quantile(0.025),
    'ci_high': boot_df.quantile(0.975),
})
print(ci_table.round(4))

ci_path = f'{DRIVE_DIR}/llm_judge_canonical_test_bootstrap_ci.csv'
ci_table.to_csv(ci_path)
print('\nSaved', ci_path)

meta_path = f'{DRIVE_DIR}/metric_run_metadata.csv'
new_row = {
    'metric': 'llm_judge_p_no (zero-shot Yes/No probability)',
    'model_or_method': MODEL_ID,
    'device': 'cuda (T4)',
    'compute_time_seconds': round(float(final['seconds'].sum()), 1),
    'val_threshold': float(best_thresh), 'val_f1': round(float(best_f1), 4),
    'test_accuracy': round(test_metrics['accuracy'], 4), 'test_precision': round(test_metrics['precision'], 4),
    'test_recall': round(test_metrics['recall'], 4), 'test_f1': round(test_metrics['f1'], 4),
    'test_roc_auc': round(test_metrics['roc_auc'], 4), 'test_pr_auc': round(test_metrics['pr_auc'], 4),
}
meta = pd.read_csv(meta_path) if os.path.exists(meta_path) else pd.DataFrame()
meta = pd.concat([meta, pd.DataFrame([new_row])], ignore_index=True)
meta.to_csv(meta_path, index=False)
print('Appended to', meta_path, '| rows now:', len(meta))


           estimate  ci_low  ci_high
accuracy     0.6162  0.5686   0.6667
precision    0.5323  0.4706   0.5957
recall       0.8627  0.8054   0.9167
f1           0.6584  0.6036   0.7129
roc_auc      0.6783  0.6211   0.7331
pr_auc       0.5395  0.4652   0.6249

Saved /content/drive/MyDrive/rag-eval-metrics/llm_judge_canonical_test_bootstrap_ci.csv
Appended to /content/drive/MyDrive/rag-eval-metrics/metric_run_metadata.csv | rows now: 4
